# Evaluate AI Text Detector

This notebook loads a saved checkpoint from the training notebook and evaluates it on the held-out test split. If the saved split file is missing, it can rebuild the split from the saved run configuration.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

from sklearn.model_selection import train_test_split

from dataset_loader import DatasetConfig, Label, Sample, load_combined_dataset
from evaluate import (
    evaluate_model,
    evaluate_per_source,
    print_eval_report,
    print_source_breakdown,
)
from inference import Detector


In [ ]:
def split_loaded_data(result, test_ratio: float, seed: int):
    indices = list(range(len(result.texts)))
    stratify = None

    source_label_groups = [
        f"{result.samples[i].source_dataset}:{result.labels[i]}"
        for i in indices
    ]
    group_counts = Counter(source_label_groups)
    if group_counts and min(group_counts.values()) >= 2:
        stratify = source_label_groups
    else:
        label_counts = Counter(result.labels)
        if label_counts and min(label_counts.values()) >= 2:
            stratify = result.labels

    try:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=stratify,
        )
    except ValueError:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=None,
        )

    test_samples = [result.samples[i] for i in test_idx]
    return test_samples


def load_samples_jsonl(path: str | Path) -> list[Sample]:
    samples: list[Sample] = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            samples.append(
                Sample(
                    text=row["text"],
                    label=Label(int(row["label"])),
                    source_dataset=row.get("source_dataset", "unknown"),
                    generator=row.get("generator", "unknown"),
                    domain=row.get("domain", "unknown"),
                )
            )
    return samples


def samples_to_texts_labels(samples: list[Sample]) -> tuple[list[str], list[int]]:
    texts = [sample.text for sample in samples]
    labels = [int(sample.label) for sample in samples]
    return texts, labels


In [ ]:
artifact_dir = Path("checkpoints/notebook_run")
run_config_path = artifact_dir / "notebook_run_config.json"
test_split_path = artifact_dir / "test_samples.jsonl"

detector = Detector.from_checkpoint(artifact_dir)

run_config = {}
if run_config_path.exists():
    run_config = json.loads(run_config_path.read_text(encoding="utf-8"))

if test_split_path.exists():
    test_samples = load_samples_jsonl(test_split_path)
else:
    if not run_config:
        raise FileNotFoundError(
            f"Missing both {test_split_path} and {run_config_path}. Run the training notebook first."
        )
    data_config = DatasetConfig(**run_config["data_config"])
    result = load_combined_dataset(data_config)
    test_samples = split_loaded_data(
        result,
        test_ratio=run_config["test_ratio"],
        seed=run_config["train_config"]["seed"],
    )

test_texts, test_labels = samples_to_texts_labels(test_samples)

print(f"Checkpoint directory: {artifact_dir}")
print(f"Test samples: {len(test_samples):,}")
print("Label distribution:", Counter(test_labels))
print("Source distribution:", Counter(sample.source_dataset for sample in test_samples))


In [ ]:
batch_size = run_config.get("train_config", {}).get("batch_size", 32)

eval_result = evaluate_model(
    detector.model,
    detector.tokenizer,
    test_texts,
    test_labels,
    batch_size=batch_size,
    threshold=detector.threshold,
)
print_eval_report(eval_result, title="Held-out Test Evaluation")


In [ ]:
breakdowns = evaluate_per_source(detector, test_samples)
if breakdowns:
    print_source_breakdown(breakdowns)
else:
    print("Per-source breakdown is unavailable for the current test split.")
